In [0]:
import requests
import zipfile
from io import BytesIO
from datetime import date

edgar_base_url = "https://www.sec.gov/files/dera/data/financial-statement-data-sets/"
volume_base_path = "/Volumes/operations/finance_staging/edgar_data"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36 (Contact: your-email@example.com)'
}

# Dynamically build year range through current year
current_year = date.today().year
years = [str(y) for y in range(2019, current_year + 1)]
quarters = ['q1', 'q2', 'q3', 'q4']

def download_and_unzip(url, extract_to):
    print(f"Downloading ZIP file from {url} ...")
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    zip_file = zipfile.ZipFile(BytesIO(response.content))
    print(f"Extracting the contents to {extract_to}...")
    zip_file.extractall(path=extract_to)
    zip_file.close()

for year in years:
    for quarter in quarters:
        edgar_url = f"{edgar_base_url}{year}{quarter}.zip"
        volume_path = f"{volume_base_path}/{year}/{quarter}/"

        try:
            download_and_unzip(url=edgar_url, extract_to=volume_path)
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 404:
                print(f"Skipping {year} {quarter.upper()} — not yet available.")
            else:
                raise  # Re-raise unexpected HTTP errors